In [1]:

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
)

import joblib

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [2]:
df_a = pd.read_parquet("/kaggle/input/datasets/mariamamin30/hate-spech-data/dynahate.parquet")

In [3]:
import pandas as pd

df_a = pd.read_parquet(
    "/kaggle/input/datasets/mariamamin30/hate-spech-data/dynahate.parquet"
)

print("Shape:", df_a.shape)
print("\nColumns:")
print(df_a.columns.tolist())

display(df_a.head())

Shape: (123432, 2)

Columns:
['text', 'label']


,text,label
0,dalits are lowlives,1
1,gay people are a burden to our society,1
2,Arabs are not welcome,1
3,I'm not saying we should actually eliminate he...,1
4,bananas are for black people,1


In [ ]:


df_clean = df_a[["text", "label"]].copy()

df_clean["label"] = pd.to_numeric(df_clean["label"],errors="coerce")

df_clean = (df_clean.dropna(subset=["text", "label"]).reset_index(drop=True))

df_clean["label"] = df_clean["label"].astype(int)

print("Shape:", df_clean.shape)

print("\nLabel distribution:")
print(df_clean["label"].value_counts(normalize=True))

display(df_clean.head())

Shape: (123432, 2)

Label distribution:
label
1    0.538961
0    0.461039
Name: proportion, dtype: float64


,text,label
0,dalits are lowlives,1
1,gay people are a burden to our society,1
2,Arabs are not welcome,1
3,I'm not saying we should actually eliminate he...,1
4,bananas are for black people,1


In [ ]:
TARGET_MODEL = "facebook/roberta-hate-speech-dynabench-r4-target"

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
target_model = AutoModelForSequenceClassification.from_pretrained(TARGET_MODEL).to(DEVICE)
target_model.eval()

print(target_model.config.id2label) 

config.json:   0%|          | 0.00/816 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: facebook/roberta-hate-speech-dynabench-r4-target
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{0: 'nothate', 1: 'hate'}


In [ ]:
@torch.no_grad()
def score_texts(texts, batch_size=32):
    scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
        logits = target_model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        hate_idx = [k for k, v in target_model.config.id2label.items() if v.lower() == "hate"][0]
        scores.extend(probs[:, hate_idx].cpu().numpy().tolist())
    return np.array(scores)


sample_texts = df_a["text"].sample(8, random_state=0).tolist()
print(score_texts(sample_texts))


[9.99780357e-01 1.34264905e-04 1.84031465e-04 1.40248841e-04
 9.99791563e-01 9.99715388e-01 1.42100849e-04 9.78827238e-01]


In [ ]:

train_df = df_a 

train_df, test_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["label"], random_state=42
)
train_df, val_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["label"], random_state=42
)

print(len(train_df), len(val_df), len(test_df))

89179 15738 18515


In [ ]:
embedder = SentenceTransformer('all-mpnet-base-v2', device=DEVICE)

def embed(texts, batch_size=64):
    return embedder.encode(
        list(texts), batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True
    )

X_train=embed(train_df["text"])
X_val=embed(val_df["text"])
X_test=embed(test_df["text"])

y_train=train_df["label"].values
y_val=val_df["label"].values
y_test=test_df["label"].values

X_train.shape, X_val.shape, X_test.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1394 [00:00<?, ?it/s]

Batches:   0%|          | 0/246 [00:00<?, ?it/s]

Batches:   0%|          | 0/290 [00:00<?, ?it/s]

((89179, 768), (15738, 768), (18515, 768))

In [9]:
clf_lr = LogisticRegression(max_iter=2000, class_weight="balanced")
clf_lr.fit(X_train, y_train)

val_probs = clf_lr.predict_proba(X_val)[:, 1]
val_preds = (val_probs > 0.5).astype(int)

print("Val accuracy:", accuracy_score(y_val, val_preds))
print("Val F1:", f1_score(y_val, val_preds))
print("Val AUC:", roc_auc_score(y_val, val_probs))
print(classification_report(y_val, val_preds))

Val accuracy: 0.7368153513788284
Val F1: 0.7538626099358212
Val AUC: 0.818337967528097
              precision    recall  f1-score   support

           0       0.71      0.72      0.72      7256
           1       0.76      0.75      0.75      8482

    accuracy                           0.74     15738
   macro avg       0.74      0.74      0.74     15738
weighted avg       0.74      0.74      0.74     15738



In [ ]:
import lightgbm as lgb
import optuna

def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, probs)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

clf_lgb = lgb.LGBMClassifier(**study.best_params)
clf_lgb.fit(X_train, y_train)

[I 2026-06-17 14:22:04,684] A new study created in memory with name: no-name-13783b34-3339-4c82-bd29-3f60feed3483


  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[I 2026-06-17 14:22:45,807] Trial 0 finished with value: 0.8552915383169548 and parameters: {'n_estimators': 187, 'learning_rate': 0.019838768905910114, 'num_leaves': 87, 'max_depth': 6, 'subsample': 0.6879533062028951, 'colsample_bytree': 0.6377961308526602}. Best is trial 0 with value: 0.8552915383169548.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[I 2026-06-17 15:45:33,425] Trial 28 finished with value: 0.9943587165713399 and parameters: {'n_estimators': 477, 'learning_rate': 0.0790505894205258, 'num_leaves': 117, 'max_depth': 10, 'subsample': 0.7043176824821552, 'colsample_bytree': 0.9997715600185915}. Best is trial 21 with value: 0.9978854306428012.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[I 2026-06-17 15:46:34,816] Trial 29 finished with value: 0.9596576621040939 and parameters: {'n_estimators': 227, 'learning_rate': 0.06284255144992619, 'num_leaves': 82, 'max_depth': 8, 'subsample': 0.6700616475728156, 'colsample_bytree': 0.6282880409406991}. Best is trial 21 with value: 0.9978854306428012.
Best AUC: 0.9978854306428012
Best params: {'n_estimators': 594, 'learning_rate': 0.16638269134626593, 'num_leaves': 128, 'max_depth': 12, 'subsample': 0.6063150911859416, 'colsample_bytree': 0.9965120652459196}


LGBMClassifier(colsample_bytree=0.9965120652459196,
               learning_rate=0.16638269134626593, max_depth=12,
               n_estimators=594, num_leaves=128, subsample=0.6063150911859416)

In [11]:
for name, model in [("Logistic Regression", clf_lr), ("LightGBM", clf_lgb)]:
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs > 0.5).astype(int)
    print(f"=== {name} ===")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("F1:", f1_score(y_test, preds))
    print("AUC:", roc_auc_score(y_test, probs))
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))
    
    print()

=== Logistic Regression ===
Accuracy: 0.7410748042128005
F1: 0.7569458527682011
AUC: 0.8235726727157959
[[6256 2280]
 [2514 7465]]
              precision    recall  f1-score   support

           0       0.71      0.73      0.72      8536
           1       0.77      0.75      0.76      9979

    accuracy                           0.74     18515
   macro avg       0.74      0.74      0.74     18515
weighted avg       0.74      0.74      0.74     18515




/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


=== LightGBM ===
Accuracy: 0.978341884958142
F1: 0.9799910184122549
AUC: 0.9984001313724145
[[8294  242]
 [ 159 9820]]
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      8536
           1       0.98      0.98      0.98      9979

    accuracy                           0.98     18515
   macro avg       0.98      0.98      0.98     18515
weighted avg       0.98      0.98      0.98     18515




In [12]:
from scipy.stats import pearsonr, spearmanr

small_scores = clf_lr.predict_proba(X_test)[:, 1]
big_scores = score_texts(test_df["text"].tolist())

print("Pearson r:", pearsonr(small_scores, big_scores))
print("Spearman r:", spearmanr(small_scores, big_scores))

Pearson r: PearsonRResult(statistic=np.float64(0.5821002678208566), pvalue=np.float64(0.0))
Spearman r: SignificanceResult(statistic=np.float64(0.6037519302900446), pvalue=np.float64(0.0))


In [13]:
from scipy.stats import pearsonr, spearmanr

small_scores = clf_lgb.predict_proba(X_test)[:, 1]
# big_scores = score_texts(test_df["text"].tolist())

print("Pearson r:", pearsonr(small_scores, big_scores))
print("Spearman r:", spearmanr(small_scores, big_scores))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Pearson r: PearsonRResult(statistic=np.float64(0.8051201831095325), pvalue=np.float64(0.0))
Spearman r: SignificanceResult(statistic=np.float64(0.7976839738852733), pvalue=np.float64(0.0))


In [ ]:
joblib.dump(clf_lr, "hate_classifier_lr.joblib")
joblib.dump(clf_lgb, "hate_classifier_lgb.joblib")
